## MarketWatch Scraper

In [18]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import quote_plus
import duckdb as dd
from datetime import datetime
import time


DB_PATH = "marketwatch_news.duckdb"


def init_db(db_path: str = DB_PATH):
    conn = dd.connect(db_path)
    conn.execute(
        '''
        DROP TABLE IF EXISTS marketwatch_articles;
        CREATE TABLE IF NOT EXISTS marketwatch_articles (
            query TEXT,
            title TEXT,
            url TEXT,
            snippet TEXT,
            source TEXT,
            published_at TIMESTAMP,
            scraped_at TIMESTAMP
        );
        '''
    )
    conn.close()


def build_search_url(query: str, page: int = 1) -> str:
    """
    Use MarketWatch search, sorted by recency.
    This targets both tickers and company names.
    """
    q = quote_plus(query)
    # Basic search URL; you can refine with more params if needed
    ## return f"https://www.marketwatch.com/search?q={q}&page={page}&sort=recency"
    return f"https://www.marketwatch.com/search?q={q}&ts=0&tab=All%20News"


def fetch_search_page(query: str, page: int = 1) -> BeautifulSoup:
    url = build_search_url(query, page)
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8',
        'Accept-Language': 'en-US,en;q=0.9',
        'Referer': 'https://www.marketwatch.com/',
        'Connection': 'keep-alive',
        'Upgrade-Insecure-Requests': '1',
    }

    session = requests.Session()
    session.headers.update(headers)

    resp = session.get(url, timeout=15)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")


def parse_search_results(soup: BeautifulSoup):
    """
    Parse MarketWatch search results page.

    NOTE: MarketWatch may change its HTML structure.
    Inspect the page in your browser and adjust selectors if needed.
    """
    articles = []

    # Common pattern: result cards under something like:
    # <div class="searchresult"> or <div class="article__content"> etc.
    # We'll try a couple of patterns.
    result_containers = soup.select("div.searchresult, div.article__content")

    for container in result_containers:
        # Title and URL
        a = container.find("a", href=True)
        if not a:
            continue

        title = a.get_text(strip=True)
        url = a["href"]

        # Snippet / summary
        snippet_el = container.find("p")
        snippet = snippet_el.get_text(strip=True) if snippet_el else None

        # Source and timestamp (if present)
        source = None
        published_at = None

        # Example patterns; adjust as needed:
        source_el = container.find("span", class_="source")
        if source_el:
            source = source_el.get_text(strip=True)

        time_el = container.find("span", class_="article__timestamp") or \
                  container.find("span", class_="timestamp")
        if time_el:
            raw_time = time_el.get_text(strip=True)
            # MarketWatch often uses relative or formatted times; you may need
            # custom parsing. For now, store raw string or try a best-effort parse.
            try:
                # Example: "Apr. 10, 2026 at 4:15 p.m. ET"
                published_at = datetime.strptime(raw_time, "%b. %d, %Y at %I:%M %p ET")
            except Exception:
                # Fallback: store as None or keep raw string in snippet if you prefer
                published_at = None

        articles.append(
            {
                "title": title,
                "url": url,
                "snippet": snippet,
                "source": source,
                "published_at": published_at,
            }
        )

    return articles


def save_articles_to_duckdb(
    query: str,
    articles,
    db_path: str = DB_PATH,
):
    conn = dd.connect(db_path)
    scraped_at = datetime.utcnow()

    for art in articles:
        conn.execute(
            """
            INSERT INTO marketwatch_articles
                (query, title, url, snippet, source, published_at, scraped_at)
            VALUES (?, ?, ?, ?, ?, ?, ?)
            """,
            [
                query,
                art.get("title"),
                art.get("url"),
                art.get("snippet"),
                art.get("source"),
                art.get("published_at"),
                scraped_at,
            ],
        )

    conn.close()


def scrape_marketwatch_for_query(
    query: str,
    pages: int = 1,
    delay_seconds: float = 1.5,
):
    """
    Scrape MarketWatch search results for a ticker or company name
    and store them into DuckDB.
    """
    all_articles = []

    for page in range(1, pages + 1):
        print(f"Fetching page {page} for query '{query}'...")
        soup = fetch_search_page(query, page)
        articles = parse_search_results(soup)
        print(f"  Found {len(articles)} articles on page {page}.")
        all_articles.extend(articles)
        time.sleep(delay_seconds)  # be polite

    save_articles_to_duckdb(query, all_articles)
    print(f"Saved {len(all_articles)} articles for query '{query}'.")


## Run Main Program

In [19]:
if __name__ == "__main__":
    # 1. Initialize DB (run once)
    init_db()

    # 2. Example: scrape daily news for a ticker
    #    You can schedule this script via cron/Task Scheduler to run daily.
    ticker = "MSFT"          # or "MSFT", "TSLA", etc.
    company_name = "Microsoft"   # you can also use company names

    # Scrape by ticker
    scrape_marketwatch_for_query(ticker, pages=2)

    # Scrape by company name (optional)
    scrape_marketwatch_for_query(company_name, pages=1)

Fetching page 1 for query 'MSFT'...
  Found 77 articles on page 1.
Fetching page 2 for query 'MSFT'...
  Found 77 articles on page 2.


C:\Users\Dsavs\AppData\Local\Temp\ipykernel_21124\4237208183.py:129: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  scraped_at = datetime.utcnow()


Saved 154 articles for query 'MSFT'.
Fetching page 1 for query 'Microsoft'...
  Found 77 articles on page 1.
Saved 77 articles for query 'Microsoft'.


## Query Results

In [21]:
## import duckdb as dd
con = dd.connect(DB_PATH)  # Open the persistent database
result = con.sql("SELECT count(*) FROM marketwatch_articles WHERE (url LIKE '%barrons.com%' or url LIKE '%marketwatch.com%')").fetchall()  # Query results
result

[(216,)]